# 02 Feature engineering

**Decision time:** the night before the flight. A feature is allowed only if
1. it is known from the published schedule (carrier, route, scheduled times, aircraft, how many flights are scheduled), or
2. it is computed from flights on **strictly earlier dates**.

Part 2 is checked by the leakage test at the end of this notebook.
Output: `data/processed/features.pkl` (all flights + features) and `feature_columns.json`.

In [1]:
import json

import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

In [2]:
from pathlib import Path

ROOT = Path("..").resolve()
RAW_CSV = ROOT / "data_raw" / "flights.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
for d in (PROCESSED_DIR, MODEL_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [3]:
DELAY_THRESHOLD = 15          # minutes, BTS definition of a delayed arrival
TRAIN_END = "2013-09-30"      # train: Jan-Sep
VAL_END = "2013-11-30"        # validation: Oct-Nov, test: Dec

# only known once the plane has left the gate, so never used as inputs
POST_DEPARTURE_COLS = ["dep_time", "dep_delay", "arr_time", "arr_delay", "air_time"]


def assign_split(dates):
    split = pd.Series("test", index=dates.index)
    split[dates <= VAL_END] = "val"
    split[dates <= TRAIN_END] = "train"
    return split


def load_raw(path=RAW_CSV):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df[["year", "month", "day"]])
    df["route"] = df["origin"] + "-" + df["dest"]
    # missing arr_delay = cancelled or diverted: kept for schedule counts, not used as examples
    df["no_arrival"] = df["arr_delay"].isna()
    df["delayed"] = (df["arr_delay"] >= DELAY_THRESHOLD).astype(int)
    df["split"] = assign_split(df["date"])
    return df.sort_values(["date", "sched_dep_time", "id"]).reset_index(drop=True)


SMOOTHING = 20   # pseudo-count used to shrink historical rates towards the system-wide rate

## Feature groups

No day-of-year or month features: with a single year of data, Oct-Dec never appear in training, so their
seasonality cannot be learned. On validation they only let XGBoost memorise particular training days.

In [4]:
CAT_COLS = ["carrier", "origin", "dest"]

CALENDAR_COLS = [
    "dep_sin", "dep_cos", "arr_sin", "arr_cos",
    "dow_sin", "dow_cos", "is_weekend", "days_to_holiday",
]
SCHEDULE_COLS = ["dep_hour", "distance", "overnight"]
CONGESTION_COLS = [
    "origin_day_flights", "origin_hour_flights", "dest_day_flights",
    "carrier_day_flights", "tail_day_flights", "tail_day_leg",
]

# (feature name, grouping column) for the long-run historical delay rates
HIST_KEYS = [
    ("carrier", "carrier"),
    ("origin", "origin"),
    ("dest", "dest"),
    ("route", "route"),
    ("carrier_route", "_carrier_route"),
    ("origin_hour", "_origin_hour"),
    ("flight", "_flight"),
    ("tailnum", "tailnum"),
]
RECENT_KEYS = ["origin", "carrier", "dest"]

LONG_HISTORY_COLS = [f"hist_rate_{name}" for name, _ in HIST_KEYS]
RECENT_HISTORY_COLS = [f"rate7_{k}" for k in RECENT_KEYS] + [
    "prev_rate_all", "prev_rate_origin", "prev_cancel_origin", "prev_mean_delay_origin",
]
HISTORY_COLS = LONG_HISTORY_COLS + RECENT_HISTORY_COLS

NUM_COLS = CALENDAR_COLS + SCHEDULE_COLS + CONGESTION_COLS + HISTORY_COLS
ALL_FEATURES = CAT_COLS + NUM_COLS
print(len(ALL_FEATURES), "features")

35 features


## Calendar and schedule features

Times of day and days of week are cyclic, so they are encoded with sine and cosine
(23:59 should be close to 00:00).

In [5]:
def cyclic(x, period):
    angle = 2 * np.pi * x / period
    return np.sin(angle), np.cos(angle)


def add_calendar_features(df):
    dep_min = df["hour"] * 60 + df["minute"]
    arr_min = (df["sched_arr_time"] // 100) * 60 + df["sched_arr_time"] % 100
    dow = df["date"].dt.day_of_week

    df["dep_sin"], df["dep_cos"] = cyclic(dep_min, 1440)
    df["arr_sin"], df["arr_cos"] = cyclic(arr_min, 1440)
    df["dow_sin"], df["dow_cos"] = cyclic(dow, 7)
    df["is_weekend"] = (dow >= 5).astype(int)

    holidays = USFederalHolidayCalendar().holidays(start="2012-11-01", end="2014-02-28")
    dates = pd.DatetimeIndex(df["date"].unique())
    gap = np.abs((dates.values[:, None] - holidays.values[None, :]).astype("timedelta64[D]").astype(int))
    df["days_to_holiday"] = df["date"].map(pd.Series(gap.min(axis=1).clip(max=14), index=dates))

    df["dep_hour"] = df["hour"]
    df["overnight"] = (df["sched_arr_time"] < df["sched_dep_time"]).astype(int)
    return df

## Congestion features

Scheduled volume is known in advance, so cancelled flights are counted too. The aircraft features say how many
NYC departures this aircraft has today and which leg this is; later legs inherit delays from earlier ones.

In [6]:
def add_congestion_features(df):
    size = lambda keys: df.groupby(keys)["id"].transform("size")
    df["origin_day_flights"] = size(["origin", "date"])
    df["origin_hour_flights"] = size(["origin", "date", "hour"])
    df["dest_day_flights"] = size(["dest", "date"])
    df["carrier_day_flights"] = size(["carrier", "date"])
    df["tail_day_flights"] = size(["tailnum", "date"])
    df["tail_day_leg"] = df.groupby(["tailnum", "date"])["sched_dep_time"].rank(method="first")
    return df

## Historical delay features (past dates only)

For a key $g$ (e.g. a route) we sum delayed and labelled flights over **earlier days only**:
$K_g(d) = \sum_{d' < d} k_g(d')$, $N_g(d) = \sum_{d' < d} n_g(d')$ (or over the last $w$ days).
In code: a daily table per key, then a cumulative / rolling sum, then `shift(1)` so the flight's own day is excluded.

Rare keys get noisy rates, so they are shrunk towards the system-wide past rate $p_0$:
$\hat r_g = (K_g + m\,p_0) / (N_g + m)$ with $m = 20$.

In [7]:
def past_sums(df, key, cols, window=None):
    """Per-flight sums of `cols` over earlier days for the same `key`.

    window=None uses all earlier days, otherwise only the last `window` days.
    The shift(1) is what keeps the flight's own day (and anything after it) out.
    """
    daily = df.groupby([key, "date"])[cols].sum()
    all_dates = pd.date_range(df["date"].min(), df["date"].max(), name="date")
    parts = {}
    for c in cols:
        wide = daily[c].unstack(key).reindex(all_dates).fillna(0)
        acc = wide.cumsum() if window is None else wide.rolling(window, min_periods=1).sum()
        parts[c] = acc.shift(1).stack()
    past = pd.DataFrame(parts)
    past.index.names = ["date", key]
    return df[["date", key]].join(past, on=["date", key])[cols]


def smoothed_rate(k, n, prior, m=SMOOTHING):
    return (k + m * prior) / (n + m)


def add_history_features(df):
    df["n_obs"] = (~df["no_arrival"]).astype(int)
    df["n_sched"] = 1
    df["delay_minutes"] = df["arr_delay"].fillna(0)
    df["_all"] = "all"
    df["_carrier_route"] = df["carrier"] + "_" + df["route"]
    df["_origin_hour"] = df["origin"] + "_" + df["hour"].astype(str)
    df["_flight"] = df["carrier"] + df["flight"].astype(str)

    cols = ["delayed", "n_obs"]

    # long-run rates, shrunk towards the system-wide rate so rare keys stay sensible
    sys_all = past_sums(df, "_all", cols)
    prior = sys_all["delayed"] / sys_all["n_obs"]
    for name, key in HIST_KEYS:
        s = past_sums(df, key, cols)
        df[f"hist_rate_{name}"] = smoothed_rate(s["delayed"], s["n_obs"], prior)

    # last 7 days
    sys7 = past_sums(df, "_all", cols, window=7)
    prior7 = sys7["delayed"] / sys7["n_obs"]
    for key in RECENT_KEYS:
        s = past_sums(df, key, cols, window=7)
        df[f"rate7_{key}"] = smoothed_rate(s["delayed"], s["n_obs"], prior7)

    # yesterday: delays tend to persist (weather systems, knock-on effects)
    sys1 = past_sums(df, "_all", cols, window=1)
    df["prev_rate_all"] = sys1["delayed"] / sys1["n_obs"]
    s = past_sums(df, "origin", ["delayed", "n_obs", "n_sched", "no_arrival", "delay_minutes"], window=1)
    df["prev_rate_origin"] = s["delayed"] / s["n_obs"]
    df["prev_cancel_origin"] = s["no_arrival"] / s["n_sched"]
    df["prev_mean_delay_origin"] = s["delay_minutes"] / s["n_obs"]

    df[HISTORY_COLS] = df[HISTORY_COLS].replace([np.inf, -np.inf], np.nan)
    return df.drop(columns=["n_obs", "n_sched", "delay_minutes", "_all",
                            "_carrier_route", "_origin_hour", "_flight"])


def build_features(df):
    df = add_calendar_features(df)
    df = add_congestion_features(df)
    df = add_history_features(df)
    return df

## Build and save

In [8]:
raw = load_raw()
df = build_features(raw.copy())
df.to_pickle(PROCESSED_DIR / "features.pkl")

with open(PROCESSED_DIR / "feature_columns.json", "w") as f:
    json.dump({
        "cat": CAT_COLS, "num": NUM_COLS,
        "groups": {"calendar": CALENDAR_COLS, "schedule": SCHEDULE_COLS, "congestion": CONGESTION_COLS,
                   "long_history": LONG_HISTORY_COLS, "recent_history": RECENT_HISTORY_COLS},
    }, f, indent=2)

print(f"saved {len(df):,} rows, {len(ALL_FEATURES)} features")
df.groupby("split")["delayed"].agg(["size", "mean"])

saved 336,776 rows, 35 features


,size,mean
split,,
test,28135,0.322161
train,252484,0.242106
val,56157,0.176434


In [9]:
obs = df[~df["no_arrival"]]
print("share missing per feature (only Jan 1 has no history):")
obs[ALL_FEATURES].isna().mean().loc[lambda s: s > 0].round(4)

share missing per feature (only Jan 1 has no history):


hist_rate_carrier          0.0025
hist_rate_origin           0.0025
hist_rate_dest             0.0025
hist_rate_route            0.0025
hist_rate_carrier_route    0.0025
hist_rate_origin_hour      0.0025
hist_rate_flight           0.0025
hist_rate_tailnum          0.0025
rate7_origin               0.0025
rate7_carrier              0.0025
rate7_dest                 0.0025
prev_rate_all              0.0025
prev_rate_origin           0.0025
prev_cancel_origin         0.0025
prev_mean_delay_origin     0.0025
dtype: float64

In [10]:
obs[HISTORY_COLS].describe().T[["mean", "min", "max"]].round(3)

,mean,min,max
hist_rate_carrier,0.246,0.030,0.610
hist_rate_origin,0.245,0.138,0.415
hist_rate_dest,0.245,0.081,0.587
hist_rate_route,0.244,0.032,0.582
hist_rate_carrier_route,0.244,0.032,0.582
hist_rate_origin_hour,0.245,0.078,0.525
hist_rate_flight,0.247,0.035,0.726
hist_rate_tailnum,0.248,0.076,0.535
rate7_origin,0.245,0.058,0.554
rate7_carrier,0.245,0.012,0.667


## Leakage test

If a feature for day D only uses earlier days, then scrambling the outcomes of day D and later must not change any
feature value on day D or before. We build the features on Jan-Feb, scramble every outcome from 10 Feb onwards, rebuild,
and compare.

In [11]:
CUTOFF = pd.Timestamp("2013-02-10")
small = raw[raw["date"] < "2013-03-01"].reset_index(drop=True)


def scramble_outcomes(df, from_date):
    df = df.copy()
    rng = np.random.default_rng(0)
    future = df["date"] >= from_date
    df.loc[future, "arr_delay"] = rng.normal(60, 80, future.sum())
    df.loc[future, "no_arrival"] = rng.random(future.sum()) < 0.3
    df.loc[future & df["no_arrival"], "arr_delay"] = np.nan
    df["delayed"] = (df["arr_delay"] >= DELAY_THRESHOLD).astype(int)
    return df


def leakage_check():
    original = build_features(small.copy())
    scrambled = build_features(scramble_outcomes(small, CUTOFF))
    upto = original["date"] <= CUTOFF
    pd.testing.assert_frame_equal(original.loc[upto, ALL_FEATURES], scrambled.loc[upto, ALL_FEATURES])
    # make sure the scramble really changed something after the cutoff
    later = original["date"] > CUTOFF
    assert not original.loc[later, "prev_rate_all"].equals(scrambled.loc[later, "prev_rate_all"])


leakage_check()
assert not set(POST_DEPARTURE_COLS) & set(ALL_FEATURES)
print("leakage test passed: no feature depends on same-day or future outcomes")

leakage test passed: no feature depends on same-day or future outcomes


Sanity check of the test itself: if we remove the `shift(1)`, the features use same-day outcomes and the test has to fail.

In [12]:
correct_past_sums = past_sums


def past_sums(df, key, cols, window=None):  # deliberately leaky version
    daily = df.groupby([key, "date"])[cols].sum()
    all_dates = pd.date_range(df["date"].min(), df["date"].max(), name="date")
    parts = {}
    for c in cols:
        wide = daily[c].unstack(key).reindex(all_dates).fillna(0)
        acc = wide.cumsum() if window is None else wide.rolling(window, min_periods=1).sum()
        parts[c] = acc.stack()  # no shift
    past = pd.DataFrame(parts)
    past.index.names = ["date", key]
    return df[["date", key]].join(past, on=["date", key])[cols]


try:
    leakage_check()
    print("PROBLEM: the leaky version was not detected")
except AssertionError as e:
    print("leaky version detected as expected:", str(e).splitlines()[0])
finally:
    past_sums = correct_past_sums

leaky version detected as expected: DataFrame.iloc[:, 20] (column name="hist_rate_carrier") are different
